# AIGC 5005 · Module 3 · Parts 4 and 5
## Project layout, dependencies, and conda environments

Companion notebook to Parts 4 and 5 of the Module 3 learning content. Every example from that part is here, ready to run, with comments explaining what to notice.

**Estimated time:** 30 minutes

**How to use it:** read each explanation, run the cell beneath it, and compare the output with what you expected *before* you ran it. The prediction is where the learning happens. Change values and run cells again; you cannot break anything.

**Kernel:** select the kernel from your `module03` conda environment (top right in VS Code or Jupyter). To make one: `conda create -n module03 python=3.11 ipykernel`, then `conda activate module03`. If VS Code offers to install `ipykernel`, accept.

## 0. Set up a scratch folder

Everything below is created inside `scratch_part45` next to this notebook, and removed by the last cell.

In [ ]:
import os
import shutil
import sys
from importlib import metadata
from pathlib import Path

SCRATCH = Path("scratch_part45").resolve()
if SCRATCH.exists():
    shutil.rmtree(SCRATCH)
SCRATCH.mkdir()
print("Working in:", SCRATCH)

## 1. Create the conventional project layout

Part 4's layout: `src/` for the package, `tests/`, `data/` split into `raw/` and `processed/`, `config/`, plus a README and a requirements file. The entry point, `main.py`, lives inside the package.

In [ ]:
project = SCRATCH / "my_project"

folders = ["src/my_project", "tests", "data/raw", "data/processed", "config"]
for folder in folders:
    (project / folder).mkdir(parents=True)

(project / "README.md").write_text("# my_project\n\nWhat it does and how to run it.\n")
(project / "environment.yml").write_text("name: my_project\nchannels:\n  - defaults\ndependencies:\n  - python=3.11\n  - pip\n")
(project / "requirements.txt").write_text("requests==2.32.3\nipykernel==7.4.0\n")
(project / "src/my_project/__init__.py").write_text("")
(project / "src/my_project/main.py").write_text("def main():\n    print(\"run me with: python -m my_project.main\")\n")
(project / "tests/test_placeholder.py").write_text("# real tests go here\n")
(project / "config/settings.example.py").write_text('API_KEY = "replace-me"\n')
print("Created", len(folders), "folders and 7 files")

### Draw the tree

A small helper that prints a folder as a tree, so you can check your own layout at a glance. Reuse it in your lab.

In [ ]:
def show_tree(folder, prefix=""):
    """Print `folder` as an indented tree (folders first, then files)."""
    entries = sorted(folder.iterdir(), key=lambda p: (p.is_file(), p.name))
    for i, entry in enumerate(entries):
        last = i == len(entries) - 1
        print(prefix + ("`-- " if last else "|-- ") + entry.name + ("/" if entry.is_dir() else ""))
        if entry.is_dir():
            show_tree(entry, prefix + ("    " if last else "|   "))


print(project.name + "/")
show_tree(project)

## 2. Keep secrets out of the repository

The real `settings.py` is never committed. Code reads secrets from **environment variables** (or a git-ignored file) and falls back to a safe message when they are missing.

In [ ]:
(project / ".gitignore").write_text("__pycache__/\n*.egg-info/\nconfig/settings.py\ndata/processed/\n")
print((project / ".gitignore").read_text())

# Read a secret from an environment variable, never from a value typed into the source code.
api_key = os.environ.get("MY_PROJECT_API_KEY")
if api_key is None:
    print("MY_PROJECT_API_KEY is not set (that is fine for this demo).")
else:
    print("API key loaded (length", len(api_key), ")")   # never print the key itself

### Try it

Reorganize a small project of your own into this layout: move the package (including its `main.py`) under `src/`, add an empty `tests/` folder with a placeholder test file, and write a short `README.md`. Compare your result with `show_tree` above.

---

## 3. Pinned requirements

A `requirements.txt` lists exact versions (`package==version`) so anyone installing from it gets what you tested with. The function below reads a requirements file and **checks it against what is installed right now**. This is the idea behind "reproducible installs".

In [ ]:
def check_requirements(path):
    """Compare each pinned line of a requirements file with the installed version."""
    for line in Path(path).read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue                                  # skip blank lines and comments
        if "==" not in line:
            print(f"UNPINNED  {line}  (no == version, so installs can drift)")
            continue
        name, wanted = line.split("==")
        try:
            installed = metadata.version(name)
        except metadata.PackageNotFoundError:
            print(f"MISSING   {name}  (pinned {wanted}, not installed)")
            continue
        status = "OK" if installed == wanted else "MISMATCH"
        print(f"{status:<9} {name}  (pinned {wanted}, installed {installed})")

### Run it on a requirements file that shows every outcome

We pin `pip` to the exact version you have (so it reports `OK`), pin `requests` to a version that is almost certainly not yours (`MISMATCH`), pin a package that does not exist (`MISSING`), and add one unpinned line (`UNPINNED`).

**Predict first:** which line gets which label?

In [ ]:
demo = SCRATCH / "demo_requirements.txt"
demo.write_text(
    "# demo file\n"
    f"pip=={metadata.version('pip')}\n"
    "requests==0.0.1\n"
    "definitely-not-a-real-package==1.0\n"
    "pandas\n"
)
check_requirements(demo)

### See what is installed

`pip freeze` is what you would run in a terminal to capture installed packages. The same information is available from Python, which lets you filter it.

In [ ]:
installed = sorted({f"{d.metadata['Name']}=={d.version}" for d in metadata.distributions()}, key=str.lower)   # a set removes duplicates
print(len(installed), "packages installed")
print("\n".join(installed[:5]), "...")        # the first few, in alphabetical order

# In a terminal you would write the whole list to a file:  pip freeze > requirements.txt
# Then trim it to only the packages YOUR project imports directly.

## 4. Conda environments

A conda environment is an isolated, project-specific copy of Python and its packages. Each project gets its own, and the **base** environment stays clean. The cell below checks which environment this notebook is running in.

**Predict first:** are you in `base`, in a project environment, or outside conda altogether?

In [ ]:
def describe_environment(prefix=None, conda_name=None):
    """Report which Python environment is running this code, and whether it is a conda environment."""
    prefix = Path(prefix or sys.prefix)
    is_conda = (prefix / "conda-meta").is_dir()          # every conda environment has a conda-meta folder
    name = conda_name or os.environ.get("CONDA_DEFAULT_ENV") or (prefix.name if is_conda else None)
    print("Python running this notebook :", sys.executable)
    print("Environment folder           :", prefix)
    print("Conda environment?           :", is_conda)
    if not is_conda:
        print("-> Not a conda environment. Select your conda kernel (top right) before continuing.")
    elif name == "base":
        print("-> You are in base. Create a project environment instead:  conda create -n module03 python=3.11 ipykernel")
    else:
        print(f"-> Good: working in the '{name}' environment.")


describe_environment()

### The commands you will actually type

Run these in a terminal (Anaconda Prompt on Windows), not in this notebook:

```
conda create -n catalogue python=3.11     # make a new environment named catalogue
conda activate catalogue                  # switch into it; the prompt shows (catalogue)
pip install -r requirements.txt           # install the pinned packages inside it
conda env list                            # every environment; * marks the active one
conda list                                # everything installed in the active environment
conda deactivate                          # leave it when you are done
```

Install with **conda first and pip second**: if both tools change the same environment, doing the conda installs first avoids hard-to-debug conflicts.

## 5. Record the environment itself with `environment.yml`

`requirements.txt` records the pip packages. `environment.yml` records the environment: its name, the Python version, and any conda packages. A classmate rebuilds it with `conda env create -f environment.yml`.

The function below reads an `environment.yml` and reports what it asks for. It uses only the standard library, so it works in any environment.

In [ ]:
def read_environment_yml(path):
    """Very small reader for a simple environment.yml: returns (name, python_pin, other_dependencies)."""
    name, python_pin, others = None, None, []
    in_dependencies = False
    for raw in Path(path).read_text().splitlines():
        line = raw.strip()
        if line.startswith("name:"):
            name = line.split(":", 1)[1].strip()
        elif line.startswith("dependencies:"):
            in_dependencies = True
        elif in_dependencies and line.startswith("- "):
            item = line[2:].strip()
            if item.startswith("python="):
                python_pin = item.split("=", 1)[1]
            else:
                others.append(item)
    return name, python_pin, others


env_file = project / "environment.yml"
print(env_file.read_text())
name, python_pin, others = read_environment_yml(env_file)
print("environment name :", name)
print("python version   :", python_pin)
print("other packages   :", others)

### Does the running Python match the file?

This is the idea behind a reproducible install: the environment you are in should match what the file asks for. Here we compare the Python version.

In [ ]:
running = f"{sys.version_info.major}.{sys.version_info.minor}"
print("file asks for Python", python_pin, "| you are running", running)
print("match:", python_pin == running)    # False is fine here: the demo file asks for 3.11 whatever you use

### Generating a starting point

In a terminal, inside the activated environment:

```
conda env export --from-history > environment.yml
```

`--from-history` lists only what you asked conda to install (not every dependency of a dependency), which keeps the file portable across operating systems. It leaves out pip packages, which is why those stay in `requirements.txt`.

### Try it

Create a conda environment for your own project, activate it, install a package, run `pip freeze > requirements.txt`, write an `environment.yml` that names the environment and its Python version, and confirm every pip line is pinned.

## Clean up

In [ ]:
shutil.rmtree(SCRATCH)
print("Scratch folder removed:", not SCRATCH.exists())